# Part 4: Empirical Analysis of Musical Structure Using Real-World Datasets

## 1. Introduction
The aim of this analysis is to investigate the relationships between measurable musical properties (audio characteristics) and their influence on song popularity, using data-driven methods. This section complements the theoretical study “Mathematics in Music” by providing empirical evidence from real-world datasets.

### Research Questions:
1. Is there a consistent relationship between the energy and loudness of songs across different datasets?
2. How do different music genres differ in terms of audio characteristics and popularity?
3. Can the popularity of songs be predicted based on measurable characteristics such as danceability, tempo, and valence?

## 2. Data Sources
The analysis below uses two independent data sources:

1. **Spotify Song Dataset (Kaggle):** A dataset containing over 114,000 songs with detailed audio characteristics such as energy, tempo, and danceability.
2. **CORGIS Music Dataset:** A collection of 10,000 songs providing metadata and additional characteristics, such as artist popularity and year of release.

*By using these two separate datasets, one can verify whether the observed musical patterns are consistent across different collectors and time periods.*

**Data Sources and Accessibility:**  
The analysis is based on two independent, publicly accessible datasets from [Kaggle](https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset?select=dataset.csv) and the [CORGIS Educational Repository](https://corgis-edu.github.io/corgis/csv/music/). The CORGIS repository is widely used for teaching and educational data analysis, as noted by university library guides such as [NC State University Libraries](https://www.lib.ncsu.edu/formats/teaching-and-learning-datasets) and [UVA Library](https://guides.lib.virginia.edu/data).  
The datasets were used for academic, non-commercial analysis in this project.

Adding libraries:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Setting the visual style for charts
sns.set_theme(style="whitegrid")
%matplotlib inline

print("Libraries imported successfully.")

Loading the files:

In [ ]:
# Loading the datasets
df_spotify = pd.read_csv('/path/to/dataset_1.csv')
df_corgis = pd.read_csv('/path/to/dataset_2.csv')

# Basic check of the data size
print(f"Spotify Dataset: {df_spotify.shape[0]} rows, {df_spotify.shape[1]} columns")
print(f"CORGIS Dataset: {df_corgis.shape[0]} rows, {df_corgis.shape[1]} columns")

Spotify Dataset: 114000 rows, 21 columns
CORGIS Dataset: 10000 rows, 35 columns


Initial inspection:

In [ ]:
# preview the first few entries to check column names and data types consistency
df_spotify.head()

In [ ]:
# preview the first few entries to check column names and data types consistency
df_corgis.head()

## 3. Data Cleaning and Preprocessing
Before beginning the analysis, the data must be cleaned. This involves checking for missing values, duplicates and inconsistencies in the data.

In [ ]:
# Checking for missing values in Spotify dataset
print("Missing values in Spotify dataset:")
print(df_spotify.isnull().sum().sort_values(ascending=False).head(5))

print("\nMissing values in CORGIS dataset:")
# Focusing on core audio and metadata columns for CORGIS
print(df_corgis.isnull().sum().sort_values(ascending=False).head(5))

Removing invalid data: rows with missing names (for Spotify) will be removed, and technical errors (such as a tempo of 0) will be filtered out.

In [ ]:
# Cleaning Spotify Data
# remove rows with missing track names or artists
df_spotify_clean = df_spotify.dropna(subset=['track_name', 'artists'])

# filter out tracks with 0 tempo (likely errors or non-musical entries)
df_spotify_clean = df_spotify_clean[df_spotify_clean['tempo'] > 0]

# Cleaning CORGIS Data
# In CORGIS, filter out songs with year 0 as it's a common placeholder
df_corgis_clean = df_corgis[df_corgis['song.year'] > 0].copy()

print(f"Cleaned Spotify size: {df_spotify_clean.shape}")
print(f"Cleaned CORGIS size: {df_corgis_clean.shape}")

Check for duplicates (if the same track appears in different albums, compilations, or elsewhere)

In [ ]:
# check for duplicates based on track ID
duplicates_count = df_spotify_clean.duplicated(subset=['track_id']).sum()
print(f"Number of duplicate Track IDs in Spotify: {duplicates_count}")

# keep the first occurrence
df_spotify_clean = df_spotify_clean.drop_duplicates(subset=['track_id'])
print(f"Final dataset size after removing duplicates: {df_spotify_clean.shape}")

Selecting key columns for analysis (creating a list of columns needed for the analysis)

In [ ]:
# Selecting essential columns for analysis from Spotify
# these represent the 'mathematical' audio features
spotify_features = ['popularity', 'duration_ms', 'danceability', 'energy',
                   'loudness', 'speechiness', 'acousticness', 'instrumentalness',
                   'liveness', 'valence', 'tempo', 'track_genre']

df_analysis = df_spotify_clean[spotify_features].copy()

df_analysis.head()

## 4. Exploratory Data Analysis

### 4.1 Distribution of Musical Features

Understanding the distribution of musical features is essential for identifying patterns and variability in music. These distributions reflect how frequently certain musical features occur in different songs.

In [ ]:
features = ['energy', 'danceability', 'valence', 'tempo', 'loudness']

df_analysis[features].hist(bins=30, figsize=(12, 8))
plt.suptitle("Distribution of Key Musical Features")
plt.show()

**Interpretation:**

- **Energy:** The histogram shows a clear increase in frequency toward higher values, with the largest concentration of songs in the **0.7 to 1.0 range**.  
  This indicates that the dataset is strongly dominated by high-energy tracks.  
  *(The bars steadily increase from left to right, with the tallest bars located near 0.8–1.0, showing a right-skewed distribution.)*

- **Danceability:** The distribution is approximately **bell-shaped**, with a peak around **0.5–0.7**.  
  This suggests that most songs have moderate to high danceability, while very low and very high values are less common.  
  *(The histogram has a clear central peak and gradually decreases toward both ends.)*

- **Valence:** The values are relatively **evenly distributed**, with a slight concentration in the mid-range (around 0.3–0.6).  
  This indicates a wide variety of emotional tones in the dataset.  
  *(The histogram does not show a sharp peak but rather a relatively flat distribution compared to other features.)*

- **Tempo:** The distribution shows a strong cluster around **100–140 BPM**, with a peak near **120 BPM**.  
  This corresponds to typical tempos in popular music.  
  *(The histogram shows a clear peak in this range, with frequency decreasing outside this range.)*

- **Loudness:** Values are heavily concentrated between approximately **-12 dB and -5 dB**, with a strong peak around **-8 to -6 dB**.  
  This suggests that most songs are produced within a narrow range of loudness.  
  *(The histogram has a sharp peak with a narrow distribution, indicating low variability.)*

**Conclusion from Distributions:**
The distributions are clearly non-uniform and show strong clustering in specific ranges. This confirms that musical features follow structured patterns rather than random variation, and **the variables exhibit clear, non-random behavior.**


### 4.2 Relationships Among Musical Characteristics

To investigate the relationships among musical characteristics, the following correlations are calculated.

In [ ]:
corr_features = ['popularity', 'danceability', 'energy', 'loudness', 'speechiness',
                 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

plt.figure(figsize=(10, 6))
sns.heatmap(df_analysis[corr_features].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix of Key Musical Features")
plt.show()

**Interpretation:**

- A **strong positive correlation** is observed between **energy and loudness (r = 0.76)**.  
  This means that louder tracks tend to also have higher energy levels.  
  *(This is one of the highest correlation values in the matrix.)*

- A **strong negative correlation** exists between **energy and acousticness (r = -0.74)**.  
  This indicates that more acoustic tracks tend to have lower energy, while highly energetic tracks are less acoustic.  
  *(The value is close to -1, showing a strong inverse relationship.)*

- **Danceability and valence (r = 0.49)** show a moderate positive relationship.  
  This suggests that more danceable songs are often perceived as more positive or "happy".  
  *(This is one of the strongest moderate correlations in the matrix.)*

- **Loudness and acousticness (r = -0.58)** are moderately negatively correlated.  
  This means that acoustic tracks tend to be quieter, while louder tracks are typically more produced or electronic.  

- **Popularity shows very weak correlations** with all features (values close to 0).  
  This suggests that no single musical feature strongly determines how popular a song is.  
  *(Аll correlations with popularity are between approximately -0.13 and 0.07.)*
  
- Other relationships (e.g. tempo, liveness, speechiness) show weak correlations, indicating they have a smaller or more complex role in defining musical characteristics.

**Conclusion:**
The correlation matrix shows that some musical properties (such as energy, loudness, and acousticness) are strongly related, while others (like popularity) are not directly explained by these features. This confirms that music is a multidimensional system where multiple variables interact rather than a single factor determining outcomes.

### 4.3 Energy vs. Sound Intensity

From a physical standpoint, sound intensity is related to the amplitude of sound waves, while energy reflects the perceived intensity. Their relationship is illustrated visually.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(x='loudness', y='energy', data=df_analysis, alpha=0.3)
plt.title("Energy vs Loudness")
plt.show()

**Interpretation:**

The scatter plot shows a clear positive correlation between sound level and energy: as the sound level increases (values approach 0 dB), the energy also increases.

- At very low sound levels (around -40 to -30 dB), almost all songs have very low energy (close to 0).
- As the sound level increases to around -20 to -10 dB, the energy values become more scattered, but still tend to increase.
- At higher volume levels (above -10 dB), most songs have high energy values (above 0.6–0.7).

*(Evidence: The points form an upward curve, creating a curved, rising shape from the lower left corner to the upper right corner.)*

This pattern suggests that volume is a strong indicator of perceived energy, but the relationship is not entirely linear—there is greater variability in the mid-range.

From a physical perspective:
- Sound intensity is related to the amplitude of sound waves.
- Higher amplitude typically results in a higher perceived intensity (energy).

**Conclusion:**
The visual curve confirms that louder songs are more energetic, which supports the relationship between the physical properties of sound and perceived musical intensity.